In [ ]:
import glob, re, joblib
import numpy as np
import scanpy as sc
import seaborn as sns
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import string
import pyarrow.parquet as pq
import matplotlib.patches as mpatches
from Utils import colors_to_use, colors_to_use_pastel
import math
from statannotations.Annotator import Annotator
from scipy.stats import mannwhitneyu
from XplainTDA_PH import XplainTDA

# SuppFig1

In [ ]:
# --------------------------------------------------
# Settings / style
# --------------------------------------------------
datasets_aliases = {
    "PBMC": "PBMC",
    "HumanBrain": "Brain",
    "BMMCMultiOme": "BMMC",
}
datasets = list(datasets_aliases)
omic = "GEX"
n_subsets = 5

homology_palette = {
    "H0": colors_to_use[5],
    "H1": colors_to_use[14],
    "H2": colors_to_use[11],
}

plt.rcParams.update({
    "axes.labelsize": 17,
    "axes.titlesize": 17,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "lines.linewidth": 3,
    "figure.dpi": 300,
})


# --------------------------------------------------
# Load TDA objects
# --------------------------------------------------
tdas = {
    dataset: [
        XplainTDA.load(
            Path("Datasets") / dataset / "XplainTDA" / omic / "Stability" / "GRAE"
            / "Subsampling" / "subsets" / f"subset_{i}" / f"tda_subset_{i}.joblib"
        )
        for i in range(n_subsets)
    ]
    for dataset in datasets
}

# finite diagrams only
dgms = {
    dataset: {
        dim: [
            np.asarray(tda.dgms[dim])[np.isfinite(np.asarray(tda.dgms[dim])).all(axis=1)]
            for tda in tdas[dataset]
        ]
        for dim in range(3)
    }
    for dataset in datasets
}

# one common limit per homology dimension across all datasets/subsets
lims = {
    dim: 1.05 * max(
        (
            d.max()
            for dataset in datasets
            for d in dgms[dataset][dim]
            if len(d)
        ),
        default=1.0,
    )
    for dim in range(3)
}


# --------------------------------------------------
# Plot: 9 rows x 5 columns
# rows 0-2 = H0, rows 3-5 = H1, rows 6-8 = H2
# --------------------------------------------------
n_rows = 3 * len(datasets)
fig, axes = plt.subplots(n_rows, n_subsets, figsize=(16, 24))

for dim in range(3):
    color = homology_palette[f"H{dim}"]
    lim = lims[dim]

    for row, dataset in enumerate(datasets):
        grid_row = dim * len(datasets) + row

        for subset_id in range(n_subsets):
            ax = axes[grid_row, subset_id]
            dgm = dgms[dataset][dim][subset_id]

            if len(dgm):
                ax.scatter(
                    dgm[:, 0], dgm[:, 1],
                    s=18, color=color, edgecolor="none"
                )

            ax.plot([0, lim], [0, lim], "--", color="grey", lw=1.5, zorder=0)
            ax.set(xlim=(-.025, lim), ylim=(-.025, lim), aspect="equal")
            ax.spines[["top", "right"]].set_visible(False)
            ax.grid(False)

            # titles only on first row of each H block
            if row == 0:
                ax.set_title(f"Subset {subset_id}", fontsize=15, pad=6)

            # x label only on bottom row of whole figure
            ax.set_xlabel("Birth" if grid_row == n_rows - 1 else "")

            # y label only on first column
            if subset_id == 0:
                ax.set_ylabel(f"H{dim} — {datasets_aliases[dataset]}\nDeath")
            else:
                ax.set_ylabel("")

            # panel letters on first column
            if subset_id == 0:
                ax.text(
                    -0.28, 1.08,
                    rf"$\mathbf{{{string.ascii_uppercase[grid_row]}}}$",
                    transform=ax.transAxes,
                    fontsize=20,
                    va="bottom",
                    ha="left",
                )

# legend
handles = [
    Line2D(
        [0], [0], marker="o", linestyle="", markersize=8,
        markerfacecolor=homology_palette[f"H{dim}"],
        markeredgecolor="none", label=rf"$H_{dim}$"
    )
    for dim in range(3)
]

axes[0, -1].legend(
    handles=handles,
    bbox_to_anchor=(1.02, .5),
    loc="center left",
    fontsize=17,
    title_fontsize=20,
    frameon=False,
)

plt.tight_layout()
fig.savefig(
    f"Figures/XplainTDA_SuppFig1.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.3,
)
plt.show()

# SuppFig2

In [ ]:
# --------------------------------------------------
# Settings / style
# --------------------------------------------------
datasets_aliases = {
    "PBMC": "PBMC",
    "HumanBrain": "Brain",
    "BMMCMultiOme": "BMMC",
}
datasets = list(datasets_aliases)
omic = "Peak"
n_subsets = 5

homology_palette = {
    "H0": colors_to_use[5],
    "H1": colors_to_use[14],
    "H2": colors_to_use[11],
}

plt.rcParams.update({
    "axes.labelsize": 17,
    "axes.titlesize": 17,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "lines.linewidth": 3,
    "figure.dpi": 300,
})


# --------------------------------------------------
# Load TDA objects
# --------------------------------------------------
tdas = {
    dataset: [
        XplainTDA.load(
            Path("Datasets") / dataset / "XplainTDA" / omic / "Stability" / "GRAE"
            / "Subsampling" / "subsets" / f"subset_{i}" / f"tda_subset_{i}.joblib"
        )
        for i in range(n_subsets)
    ]
    for dataset in datasets
}

# finite diagrams only
dgms = {
    dataset: {
        dim: [
            np.asarray(tda.dgms[dim])[np.isfinite(np.asarray(tda.dgms[dim])).all(axis=1)]
            for tda in tdas[dataset]
        ]
        for dim in range(3)
    }
    for dataset in datasets
}

# one common limit per homology dimension across all datasets/subsets
lims = {
    dim: 1.05 * max(
        (
            d.max()
            for dataset in datasets
            for d in dgms[dataset][dim]
            if len(d)
        ),
        default=1.0,
    )
    for dim in range(3)
}


# --------------------------------------------------
# Plot: 9 rows x 5 columns
# rows 0-2 = H0, rows 3-5 = H1, rows 6-8 = H2
# --------------------------------------------------
n_rows = 3 * len(datasets)
fig, axes = plt.subplots(n_rows, n_subsets, figsize=(16, 24))

for dim in range(3):
    color = homology_palette[f"H{dim}"]
    lim = lims[dim]

    for row, dataset in enumerate(datasets):
        grid_row = dim * len(datasets) + row

        for subset_id in range(n_subsets):
            ax = axes[grid_row, subset_id]
            dgm = dgms[dataset][dim][subset_id]

            if len(dgm):
                ax.scatter(
                    dgm[:, 0], dgm[:, 1],
                    s=18, color=color, edgecolor="none"
                )

            ax.plot([0, lim], [0, lim], "--", color="grey", lw=1.5, zorder=0)
            ax.set(xlim=(-.025, lim), ylim=(-.025, lim), aspect="equal")
            ax.spines[["top", "right"]].set_visible(False)
            ax.grid(False)

            # titles only on first row of each H block
            if row == 0:
                ax.set_title(f"Subset {subset_id}", fontsize=15, pad=6)

            # x label only on bottom row of whole figure
            ax.set_xlabel("Birth" if grid_row == n_rows - 1 else "")

            # y label only on first column
            if subset_id == 0:
                ax.set_ylabel(f"H{dim} — {datasets_aliases[dataset]}\nDeath")
            else:
                ax.set_ylabel("")

            # panel letters on first column
            if subset_id == 0:
                ax.text(
                    -0.28, 1.08,
                    rf"$\mathbf{{{string.ascii_uppercase[grid_row]}}}$",
                    transform=ax.transAxes,
                    fontsize=20,
                    va="bottom",
                    ha="left",
                )

# legend
handles = [
    Line2D(
        [0], [0], marker="o", linestyle="", markersize=8,
        markerfacecolor=homology_palette[f"H{dim}"],
        markeredgecolor="none", label=rf"$H_{dim}$"
    )
    for dim in range(3)
]

axes[0, -1].legend(
    handles=handles,
    bbox_to_anchor=(1.02, .5),
    loc="center left",
    fontsize=17,
    title_fontsize=20,
    frameon=False,
)

plt.tight_layout()
fig.savefig(
    f"Figures/XplainTDA_SuppFig2.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.3,
)
plt.show()